In [5]:
import pandas as pd
import sqlite3

In [16]:
df = pd.read_csv("../googleplaystore.csv")
conn = sqlite3.connect("games.db")
df.to_sql("games", conn, if_exists="replace", index=False)

10841

In [30]:
result = pd.read_sql_query("SELECT * FROM games", conn)
result

,App,Category,Rating,Reviews,Size,Installs,Type,Price,Content Rating,Genres,Last Updated,Current Ver,Android Ver
0,Photo Editor & Candy Camera & Grid & ScrapBook,ART_AND_DESIGN,4.1,159,19M,"10,000+",Free,0,Everyone,Art & Design,"January 7, 2018",1.0.0,4.0.3 and up
1,Coloring book moana,ART_AND_DESIGN,3.9,967,14M,"500,000+",Free,0,Everyone,Art & Design;Pretend Play,"January 15, 2018",2.0.0,4.0.3 and up
2,"U Launcher Lite – FREE Live Cool Themes, Hide ...",ART_AND_DESIGN,4.7,87510,8.7M,"5,000,000+",Free,0,Everyone,Art & Design,"August 1, 2018",1.2.4,4.0.3 and up
3,Sketch - Draw & Paint,ART_AND_DESIGN,4.5,215644,25M,"50,000,000+",Free,0,Teen,Art & Design,"June 8, 2018",Varies with device,4.2 and up
4,Pixel Draw - Number Art Coloring Book,ART_AND_DESIGN,4.3,967,2.8M,"100,000+",Free,0,Everyone,Art & Design;Creativity,"June 20, 2018",1.1,4.4 and up
...,...,...,...,...,...,...,...,...,...,...,...,...,...
10836,Sya9a Maroc - FR,FAMILY,4.5,38,53M,"5,000+",Free,0,Everyone,Education,"July 25, 2017",1.48,4.1 and up
10837,Fr. Mike Schmitz Audio Teachings,FAMILY,5.0,4,3.6M,100+,Free,0,Everyone,Education,"July 6, 2018",1.0,4.1 and up
10838,Parkinson Exercices FR,MEDICAL,NaN,3,9.5M,"1,000+",Free,0,Everyone,Medical,"January 20, 2017",1.0,2.2 and up
10839,The SCP Foundation DB fr nn5n,BOOKS_AND_REFERENCE,4.5,114,Varies with device,"1,000+",Free,0,Mature 17+,Books & Reference,"January 19, 2015",Varies with device,Varies with device


To identify which genres have the most reviews and find the most popular genre.

In [28]:
result = pd.read_sql_query("SELECT App, Reviews, Genres FROM games ORDER BY Reviews DESC LIMIT 10", conn)
result

,App,Reviews,Genres
0,GollerCepte Live Score,9992,Sports
1,Ad Block REMOVER - NEED ROOT,999,Tools
2,SnipSnap Coupon App,9975,Shopping
3,SnipSnap Coupon App,9975,Shopping
4,US Open Tennis Championships 2018,9971,Sports
5,US Open Tennis Championships 2018,9971,Sports
6,DreamTrips,9971,Travel & Local
7,Adult Color by Number Book - Paint Mandala Pages,997,Entertainment
8,BSPlayer ARMv7 VFP CPU support,9966,Video Players & Editors
9,"Easy Resume Builder, Resume help, Curriculum v...",996,Tools


The results are not sorted in descending order as expected. This may be because the Reviews column is stored as a string rather than an integer. Let's verify the data type:

In [32]:
print(df['Reviews'].dtype)

object


Object is the text data type in pandas. So, the review numbers are stored as strings rather than integers. That's why the results are not sorted in descending order. Let's check whether any rows contain non-numeric values.

In [34]:
df[~df['Reviews'].str.isnumeric()]['Reviews'].unique()

array(['3.0M'], dtype=object)

One non-numeric value was found: '3.0M'. This needs to be handled before converting the column to integer.

In [40]:
df['Reviews'] = df['Reviews'].replace('3.0M', '3000000')
df['Reviews'] = df['Reviews'].astype(int)
df.to_sql("games", conn, if_exists="replace", index=False)

10841

In [41]:
result = pd.read_sql_query("SELECT App, Reviews, Genres FROM games ORDER BY Reviews DESC LIMIT 10", conn)
result

,App,Reviews,Genres
0,Facebook,78158306,Social
1,Facebook,78128208,Social
2,WhatsApp Messenger,69119316,Communication
3,WhatsApp Messenger,69119316,Communication
4,WhatsApp Messenger,69109672,Communication
5,Instagram,66577446,Social
6,Instagram,66577313,Social
7,Instagram,66577313,Social
8,Instagram,66509917,Social
9,Messenger – Text and Video Chat for Free,56646578,Communication


The results are now sorted in descending order correctly. However, we can see repeated app names. There should not be duplicate entries. Let's investigate these duplicates.

In [43]:
df[df.duplicated(subset='App', keep=False)]['App'].value_counts()

App
ROBLOX                                                9
CBS Sports App - Scores, News, Stats & Watch Live     8
Candy Crush Saga                                      7
8 Ball Pool                                           7
ESPN                                                  7
                                                     ..
Apartments & Rentals - Zillow                         2
Realtor.com Real Estate: Homes for Sale and Rent      2
Trulia Real Estate & Rentals                          2
Apartment List: Housing, Apt, and Property Rentals    2
Maps & GPS Navigation — OsmAnd                        2
Name: count, Length: 798, dtype: int64

There are 798 duplicate app entries. To keep the most representative record for each app, we will retain the entry with the highest review count, as it reflects the most popular version.

In [45]:
df = df.sort_values('Reviews', ascending=False).drop_duplicates(subset='App', keep='first')
df.to_sql("games", conn, if_exists="replace", index=False)

9660

In [46]:
result = pd.read_sql_query("SELECT App, Reviews, Genres FROM games ORDER BY Reviews DESC LIMIT 10", conn)
result

,App,Reviews,Genres
0,Facebook,78158306,Social
1,WhatsApp Messenger,69119316,Communication
2,Instagram,66577446,Social
3,Messenger – Text and Video Chat for Free,56646578,Communication
4,Clash of Clans,44893888,Strategy
5,Clean Master- Space Cleaner & Antivirus,42916526,Tools
6,Subway Surfers,27725352,Arcade
7,YouTube,25655305,Video Players & Editors
8,"Security Master - Antivirus, VPN, AppLock, Boo...",24900999,Tools
9,Clash Royale,23136735,Strategy
